# Citi Bike + Weather Merge Notebook

This notebook rebuilds the station-hour training table from raw Citi Bike trip CSVs and merges hourly Open-Meteo weather data.

It is designed to run after the remaining 2025 trip files finish downloading. It does not modify any existing notebooks.

In [2]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

PROJECT_ROOT = Path.cwd()
TRIP_ROOT = PROJECT_ROOT / 'data' / 'citibike'
WEATHER_PATH = PROJECT_ROOT / 'data' / 'weather' / 'open-meteo-2023-2025.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'proceed'

# Adjust this if you want to restrict the load to a subset of years.
TARGET_YEARS = {2023, 2024}

OUTPUT_BASENAME = 'micro_mobility_training_data_2023_2025_weather'
OUTPUT_PATH = OUTPUT_DIR / f'{OUTPUT_BASENAME}.csv'

TRIP_COLUMNS = [
    'ride_id', 'rideable_type', 'started_at', 'ended_at',
    'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id',
    'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual'
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

WindowsPath('d:/AIT/ML/project/citibike-netdemand-prediction')

In [3]:
def discover_trip_files(root: Path, years: set[int]) -> list[Path]:
    files = []
    for path in sorted(root.rglob('*.csv')):
        month_token = path.name[:6]
        if len(month_token) == 6 and month_token.isdigit():
            year = int(month_token[:4])
            if year in years:
                files.append(path)
    return files


def read_trip_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=TRIP_COLUMNS)
    df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
    df['ended_at'] = pd.to_datetime(df['ended_at'], errors='coerce')
    return df


trip_files = discover_trip_files(TRIP_ROOT, TARGET_YEARS)
print(f'Found {len(trip_files)} trip CSV files')
print('First 5 files:')
for path in trip_files[:5]:
    print(' -', path.relative_to(PROJECT_ROOT))

if not trip_files:
    raise FileNotFoundError('No trip CSV files found for TARGET_YEARS')

Found 90 trip CSV files
First 5 files:
 - data\citibike\202301-citibike-tripdata\202301-citibike-tripdata_1.csv
 - data\citibike\202301-citibike-tripdata\202301-citibike-tripdata_2.csv
 - data\citibike\202302-citibike-tripdata\202302-citibike-tripdata_1.csv
 - data\citibike\202302-citibike-tripdata\202302-citibike-tripdata_2.csv
 - data\citibike\202303-citibike-tripdata\202303-citibike-tripdata_1.csv


In [4]:
trip_frames = []

for i, path in enumerate(trip_files, start=1):
    print(f'[{i}/{len(trip_files)}] Loading {path.name}')
    trip_frames.append(read_trip_file(path))

combined_df = pd.concat(trip_frames, ignore_index=True)
del trip_frames
gc.collect()

print('Combined shape:', combined_df.shape)
combined_df.head()

[1/90] Loading 202301-citibike-tripdata_1.csv


C:\Users\thett\AppData\Local\Temp\ipykernel_7008\751560283.py:13: DtypeWarning: Columns (0: start_station_id, 1: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=TRIP_COLUMNS)


[2/90] Loading 202301-citibike-tripdata_2.csv


C:\Users\thett\AppData\Local\Temp\ipykernel_7008\751560283.py:13: DtypeWarning: Columns (0: start_station_id, 1: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=TRIP_COLUMNS)


[3/90] Loading 202302-citibike-tripdata_1.csv


...

C:\Users\thett\AppData\Local\Temp\ipykernel_7008\751560283.py:13: DtypeWarning: Columns (0: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=TRIP_COLUMNS)


[89/90] Loading 202412-citibike-tripdata_2.csv


C:\Users\thett\AppData\Local\Temp\ipykernel_7008\751560283.py:13: DtypeWarning: Columns (0: start_station_id, 1: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=TRIP_COLUMNS)


[90/90] Loading 202412-citibike-tripdata_3.csv
Combined shape: (79410195, 13)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,A8518A6C4BE513DE,classic_bike,2023-01-03 23:14:52.325,2023-01-03 23:33:42.737,E 1 St & Bowery,5636.13,Spruce St & Nassau St,5137.10,40.724861,-73.992131,40.711464,-74.005524,casual
1,A3911E4F5B9B5773,electric_bike,2023-01-07 07:57:40.054,2023-01-07 08:01:27.330,E 1 St & Bowery,5636.13,Ave A & E 11 St,5703.13,40.724861,-73.992131,40.728547,-73.981759,casual
2,AE7F74C32AEBF6F2,electric_bike,2023-01-09 18:37:44.830,2023-01-09 18:48:56.233,1 Ave & E 39 St,6303.01,E 14 St & 1 Ave,5779.10,40.747140,-73.971130,40.731393,-73.982867,member
3,6E10997509D2B7F6,electric_bike,2023-01-05 19:06:15.350,2023-01-05 19:08:33.547,E Burnside Ave & Ryer Ave,8397.02,E Burnside Ave & Ryer Ave,8397.02,40.850535,-73.901318,40.850535,-73.901318,casual
4,AA546E74A9330BD4,electric_bike,2023-01-02 20:25:23.300,2023-01-03 10:51:25.164,Clermont Ave & Park Ave,4692.01,Clermont Ave & Park Ave,4692.01,40.695734,-73.971297,40.695734,-73.971297,casual


In [5]:
# Cleaning aligned with the previous project logic.
combined_df = combined_df.dropna(subset=['started_at', 'ended_at'])
combined_df['duration_mins'] = (combined_df['ended_at'] - combined_df['started_at']).dt.total_seconds() / 60.0

clean_df = combined_df.loc[
    (combined_df['duration_mins'] >= 1) &
    (combined_df['duration_mins'] <= 1440)
].copy()


In [6]:
# Remove rows with missing station information that would break inventory accounting.
clean_df = clean_df.dropna(subset=['start_station_name', 'end_station_name'])


In [7]:
# Hour buckets for departures and arrivals.
clean_df['start_hour_ts'] = clean_df['started_at'].dt.floor('h')
clean_df['end_hour_ts'] = clean_df['ended_at'].dt.floor('h')

print('Clean shape:', clean_df.shape)
print('Trip date range:', clean_df['started_at'].min(), 'to', clean_df['started_at'].max())

Clean shape: (79173411, 16)
Trip date range: 2022-12-31 08:48:22.990000 to 2024-12-31 23:57:46.390000


In [8]:
# Coordinate lookup from both start and end station records.
start_coords = clean_df[['start_station_name', 'start_lat', 'start_lng']].rename(
    columns={'start_station_name': 'station', 'start_lat': 'lat', 'start_lng': 'lng'}
)
end_coords = clean_df[['end_station_name', 'end_lat', 'end_lng']].rename(
    columns={'end_station_name': 'station', 'end_lat': 'lat', 'end_lng': 'lng'}
)

coords_lookup = (
    pd.concat([start_coords, end_coords], ignore_index=True)
    .dropna(subset=['station', 'lat', 'lng'])
    .groupby('station', as_index=False)[['lat', 'lng']]
    .mean()
)

coords_lookup.head()

ArrowMemoryError: realloc of size 6356471808 failed

In [ ]:
# Station-hour inflow/outflow aggregation.
outflows = (
    clean_df.groupby(['start_station_name', 'start_hour_ts'])
    .size()
    .reset_index(name='outflow')
    .rename(columns={'start_station_name': 'station', 'start_hour_ts': 'datetime_hour'})
)

inflows = (
    clean_df.groupby(['end_station_name', 'end_hour_ts'])
    .size()
    .reset_index(name='inflow')
    .rename(columns={'end_station_name': 'station', 'end_hour_ts': 'datetime_hour'})
)

net_flow_df = pd.merge(outflows, inflows, on=['station', 'datetime_hour'], how='outer').fillna(0)
net_flow_df['net_demand'] = net_flow_df['inflow'] - net_flow_df['outflow']

del outflows, inflows
gc.collect()

print('Station-hour observed rows:', len(net_flow_df))
net_flow_df.head()

In [ ]:
# Reindex to a full station x hour grid so zero-activity hours are explicit.
stations = np.sort(coords_lookup['station'].unique())
full_hour_index = pd.date_range(
    start=clean_df['started_at'].min().floor('D'),
    end=clean_df['started_at'].max().ceil('D') - pd.Timedelta(hours=1),
    freq='h'
)

full_index = pd.MultiIndex.from_product(
    [stations, full_hour_index],
    names=['station', 'datetime_hour']
)

feature_df = full_index.to_frame(index=False)

final_df = (
    feature_df
    .merge(net_flow_df[['station', 'datetime_hour', 'net_demand']], on=['station', 'datetime_hour'], how='left')
    .fillna({'net_demand': 0})
    .merge(coords_lookup, on='station', how='left')
)

del feature_df, net_flow_df
gc.collect()

print('Full station-hour rows:', len(final_df))
final_df.head()

In [ ]:
# Time features and demand lags.
final_df['date'] = final_df['datetime_hour'].dt.normalize()
final_df['hour'] = final_df['datetime_hour'].dt.hour
final_df['day_of_week'] = final_df['datetime_hour'].dt.dayofweek

final_df['hour_sin'] = np.sin(2 * np.pi * final_df['hour'] / 24)
final_df['hour_cos'] = np.cos(2 * np.pi * final_df['hour'] / 24)
final_df['day_sin'] = np.sin(2 * np.pi * final_df['day_of_week'] / 7)
final_df['day_cos'] = np.cos(2 * np.pi * final_df['day_of_week'] / 7)

final_df = final_df.sort_values(['station', 'datetime_hour']).reset_index(drop=True)

for lag in [1, 2, 3, 24]:
    final_df[f'lag_{lag}h'] = final_df.groupby('station')['net_demand'].shift(lag).fillna(0)

final_df['rolling_mean_3h'] = (
    final_df.groupby('station')['net_demand']
    .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
    .fillna(0)
)

final_df.head()

In [ ]:
# Weather loading. The Open-Meteo CSV has metadata rows before the actual header.
weather_df = pd.read_csv(WEATHER_PATH, skiprows=3)
weather_df['datetime_hour'] = pd.to_datetime(weather_df['time'], errors='coerce')
weather_df = weather_df.dropna(subset=['datetime_hour']).copy()

weather_df = weather_df.rename(columns={
    'temperature_2m (°C)': 'temp_2m',
    'relative_humidity_2m (%)': 'rh_2m',
    'rain (mm)': 'rain_mm',
    'snowfall (cm)': 'snow_cm',
    'wind_speed_10m (km/h)': 'wind_kmh',
    'precipitation (mm)': 'precip_mm',
    'cloud_cover (%)': 'cloud_cover',
    'cloud_cover_low (%)': 'cloud_low',
    'cloud_cover_mid (%)': 'cloud_mid',
    'cloud_cover_high (%)': 'cloud_high'
})

weather_cols = [
    'datetime_hour', 'temp_2m', 'rh_2m', 'rain_mm', 'snow_cm',
    'wind_kmh', 'precip_mm', 'cloud_cover', 'cloud_low', 'cloud_mid', 'cloud_high'
]

weather_df = weather_df[weather_cols].sort_values('datetime_hour').reset_index(drop=True)
weather_df = weather_df.loc[
    (weather_df['datetime_hour'] >= final_df['datetime_hour'].min()) &
    (weather_df['datetime_hour'] <= final_df['datetime_hour'].max())
].copy()

print('Weather rows in merge range:', len(weather_df))
weather_df.head()

In [ ]:
# Merge weather at the hourly timestamp grain.
merged_df = final_df.merge(weather_df, on='datetime_hour', how='left')

# Optional simple derived weather flags.
merged_df['is_raining'] = (merged_df['precip_mm'].fillna(0) > 0).astype(int)
merged_df['is_snowing'] = (merged_df['snow_cm'].fillna(0) > 0).astype(int)

print('Merged shape:', merged_df.shape)
print('Weather null percentages:')
display((merged_df[['temp_2m', 'rh_2m', 'precip_mm', 'snow_cm', 'wind_kmh', 'cloud_cover']].isna().mean() * 100).round(4))
merged_df.head()

In [ ]:
# Validation checks.
row_count_ok = len(merged_df) == len(final_df)
dup_count = merged_df.duplicated(subset=['station', 'date', 'hour']).sum()

print('Row count preserved:', row_count_ok)
print('Duplicate station-date-hour rows:', dup_count)
print('Date range:', merged_df['date'].min(), 'to', merged_df['date'].max())
print('Stations:', merged_df['station'].nunique())

if not row_count_ok:
    raise ValueError('Row count changed after weather merge')
if dup_count != 0:
    raise ValueError('Duplicate station-date-hour rows detected after merge')

In [ ]:
# Save final output. Keep datetime_hour because it is the actual merge key and is useful for auditing.
ordered_cols = [
    'station', 'date', 'hour', 'datetime_hour', 'net_demand', 'lat', 'lng',
    'hour_sin', 'hour_cos', 'day_of_week', 'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h',
    'temp_2m', 'rh_2m', 'rain_mm', 'snow_cm', 'wind_kmh', 'precip_mm',
    'cloud_cover', 'cloud_low', 'cloud_mid', 'cloud_high', 'is_raining', 'is_snowing'
]

existing_cols = [col for col in ordered_cols if col in merged_df.columns]
merged_df = merged_df[existing_cols]

merged_df.to_csv(OUTPUT_PATH, index=False)
print('Saved merged dataset to:', OUTPUT_PATH)
print('Final shape:', merged_df.shape)